# geojson_pydantic

## Example 1: build a Feature from a Point and read it back

In [ ]:
# Example 1: a Feature built from a Point validates and keeps its coordinates and properties
from geojson_pydantic import Feature, Point

point = Point(type="Point", coordinates=(-122.4194, 37.7749))
feature = Feature(type="Feature", geometry=point, properties={"name": "San Francisco"})
assert feature.geometry.coordinates.longitude == -122.4194
assert feature.geometry.coordinates.latitude == 37.7749
assert feature.properties == {"name": "San Francisco"}
print("Example 1:", feature.geometry.type, tuple(feature.geometry.coordinates), feature.properties)

## Example 2: parse GeoJSON text and serialise it back

In [ ]:
# Example 2: model_validate_json -> model_dump_json is a round trip for a FeatureCollection
import json
from geojson_pydantic import FeatureCollection

text = json.dumps({"type": "FeatureCollection", "features": [
    {"type": "Feature", "geometry": {"type": "Point", "coordinates": [1.0, 2.0]}, "properties": {"id": 1}},
    {"type": "Feature", "geometry": {"type": "LineString", "coordinates": [[0, 0], [1, 1]]}, "properties": {"id": 2}},
]})
fc = FeatureCollection.model_validate_json(text)
assert len(fc.features) == 2 and [f.geometry.type for f in fc.features] == ["Point", "LineString"]
again = FeatureCollection.model_validate_json(fc.model_dump_json())
assert again == fc
print("Example 2:", len(fc.features), "features round-tripped:", [f.properties["id"] for f in again.features])

## Example 3: an invalid polygon is refused

In [ ]:
# Example 3: a Polygon ring must be closed and have 4+ positions; pydantic refuses the broken one
import pydantic
from geojson_pydantic import Polygon

ok = Polygon(type="Polygon", coordinates=[[(0, 0), (1, 0), (1, 1), (0, 0)]])
assert ok.exterior == [(0, 0), (1, 0), (1, 1), (0, 0)] or len(ok.exterior) == 4
try:
    Polygon(type="Polygon", coordinates=[[(0, 0), (1, 0), (1, 1), (2, 2)]])
except pydantic.ValidationError as error:
    message = str(error)
    print("Example 3: unclosed ring refused:", message.splitlines()[-1].strip()[:100])
else:
    raise AssertionError("an unclosed ring validated")

## Example 4: the __geo_interface__ and WKT views

In [ ]:
# Example 4: each geometry exposes __geo_interface__ (the Python geo protocol) and well-known text
from geojson_pydantic import LineString, MultiPoint

line = LineString(type="LineString", coordinates=[(0, 0), (10, 10), (20, 0)])
assert line.__geo_interface__["type"] == "LineString"
assert line.wkt == "LINESTRING (0.0 0.0, 10.0 10.0, 20.0 0.0)", line.wkt
mp = MultiPoint(type="MultiPoint", coordinates=[(1, 2, 3), (4, 5, 6)])
assert mp.has_z and mp.wkt.startswith("MULTIPOINT Z"), mp.wkt
print("Example 4:", line.wkt, "|", mp.wkt)

## Example 5: bounding boxes and a GeometryCollection

In [ ]:
# Example 5: a bbox is carried through validation and serialisation, and a GeometryCollection holds mixed geometries
from geojson_pydantic import GeometryCollection, Point, Polygon

square = Polygon(type="Polygon", coordinates=[[(0, 0), (2, 0), (2, 2), (0, 2), (0, 0)]], bbox=(0, 0, 2, 2))
gc = GeometryCollection(type="GeometryCollection", geometries=[Point(type="Point", coordinates=(1, 1)), square])
dumped = gc.model_dump(exclude_none=True)
assert dumped["geometries"][1]["bbox"] == (0, 0, 2, 2), dumped
assert [g.type for g in gc.geometries] == ["Point", "Polygon"] and len(gc.geometries) == 2
print("Example 5:", gc.wkt)